In [ ]:
# ---- Importing Media ---- #
from media import get_medium

SiN = get_medium(
    "si3n4",
    (0.600, 1.200),
)

SiO2 = get_medium(
    "sio2",
    (0.600, 1.200)
)

Si = get_medium(
    "si",
    (0.600, 1.200),
    tolerance = 1e-6
)

Air = get_medium(
    "air",
    (0.600, 1.200)
)

In [ ]:
# ---- Defining the Geometry of the Grarting Coupler ---- #

from geometries import grating_coupler
from utilities import sim_plotter
from gratingCoupler import GratingConfig, GratingCoupler

from pathlib import Path
import numpy as np

example_config = GratingConfig(
    wavelength_um=1.550,    # Central wavelength (μm)
    theta_deg=-12.0,        # Fiber incidence angle (degrees off-vertical)
    mfd=10.4,               # Mode Field Diameter (Standard SMF-28)
    source_x=3.8419,        # Fiber beam center X position
    source_gap=2.0,         # Distance between fiber source and top cladding

    # Substrate & Layer Geometry
    core_height=0.800,      # Core thickness (μm)
    cladding_height=2.0,    # Air/Cladding buffer height above core
    box_height=4.0,         # Buried Oxide (BOX) thickness
    substrate_height=8.0,   # Silicon handle thickness

    # Grating Teeth Dimensions
    etch_depth=0.5136,      # Partial etch depth (μm)
    period=0.8201,          # Grating pitch/period (μm)
    ff=0.5136,              # Fill factor (50% tooth, 50% groove)
    n_teeth=22,             # Total number of grating teeth
    w_grating=10.4,        # Transverse width for 3D runs

    # Source & Domain Settings
    pol_angle=np.pi/2,          # 0.0 = TE polarization
    buffer=3.0              # Domain margin buffer
)

example_grating_coupler_3d: GratingCoupler = grating_coupler.get_active_simulation(
    config=example_config,
    core_medium=SiN,
    box_medium=SiO2,
    substrate_medium=None,  
    cladding_medium=Air,    # Air-clad (or pass SiO2 for oxide clad)
    is_3d=True           
)

example_grating_coupler_2d : GratingCoupler = grating_coupler.get_active_simulation(
    config=example_config,
    core_medium=SiN,
    box_medium=SiO2,
    substrate_medium=None,
    cladding_medium=Air,
    is_3d=False
)

sim_plotter.display_3d(
    device = example_grating_coupler_3d,
    output_path = Path("example_results/grating_coupler/example_3d.png"),
    show = True,
    save = True
)

sim_plotter.display_2d(
    device=example_grating_coupler_2d,
    output_path=Path("example_results/grating_coupler/example_2d.png"),
    show = True,
    save = True
)

In [ ]:
# ---- Grating Coupler Analytics ---- #

# 1) Coupler efficiency
"""
    The path that is passed to these functions is where the simulation results
    are stored locally after being run on the Tidy3D servers. This ensures that
    if the same configuration is requested more than once, only the first request
    is passed to the Tidy3D servers, and the rest are served from the local data store.

    We have noted that Tidy3D servers do carry out a server-side search to ensure that
    no two simulations are run twice. This layer of local-searching reduces
    latency.

    PROBLEM: As of now the simulation saves are indifferent to the media used to
    build the simulation. This is likely to be corrected with an update to the
    Medium handling architecture.
"""
example_grating_coupler_2d.get_efficiency(Path("example_results/grating_coupler/"))
example_grating_coupler_3d.get_efficiency(Path("example_results/grating_coupler/"))

In [ ]:
# 2.1) Visualize fields within the 2D coupler
example_grating_coupler_2d.get_fields(Path("example_results/grating_coupler/"))

In [ ]:
# 2.2) Visualize fields within the 3D coupler
example_grating_coupler_3d.get_fields(Path("example_results/grating_coupler/"))

In [1]:
from pathlib import Path
import numpy as np
import tidy3d as td

from gratingCoupler.nested_sampler import GratingCouplerBayesianOptimizer

# Define Tidy3D Materials
SiN = td.Medium(permittivity=2.0**2)    # Core medium (~SiN n=2.0)
SiO2 = td.Medium(permittivity=1.45**2)  # BOX/Oxide medium (~SiO2 n=1.45)
Air = td.Medium(permittivity=1.0**2)   # Top Cladding medium (Air n=1.0)

# Package Fixed Physical & Domain Parameters
# These map directly to the GratingConfig dataclass inside the sampler
fixed_params = {
    "lda0": 1.550,            # Central excitation wavelength (μm)
    "theta_deg": -12.0,        # Fiber angle off-vertical (degrees)
    "mfd": 10.4,               # Mode Field Diameter of fiber source (μm)
    "source_gap": 2.0,         # Gap above cladding for source plane (μm)
    
    # Layer Heights
    "core_height": 0.800,      # Silicon Nitride core thickness (μm)
    "cladding_height": 2.0,    # Cladding height buffer (μm)
    "box_height": 4.0,         # Buried Oxide (BOX) thickness (μm)
    "substrate_height": 8.0,   # Silicon handle substrate thickness (μm)
    
    # 3D Domain & Numerical Resolution
    "w_grating": 10.4,         # Transverse width for 3D runs (μm)
    "buffer": 3.0,             # Spatial buffer margin around domain (μm)
    "pol_angle": np.pi / 2.0,  # TE polarization (90 degrees)
    "fine_steps": 15,          # Grid steps per wavelength in fine mesh region
    "coarse_steps": 10         # Grid steps per wavelength in background mesh
}

# Package Material Media Dictionary
materials = {
    "core_medium": SiN,
    "box_medium": SiO2,
    "sub_medium": None,        # Terminated at BOX layer (set Si medium to include handle)
    "cladding_medium": Air     # Cladding medium (Air or SiO2)
}

# Define Tight Prior Search Envelopes Around Design Point
# Constraining the prior space guarantees high likelihood values early in the run
tight_param_bounds = {
    "period": [0.800, 0.840],       # Centered near Lambda = 0.8201 μm
    "fill_factor": [0.480, 0.550],  # Centered near Duty Cycle = 0.5136
    "etch_depth": [0.480, 0.550],   # Centered near Etch Depth = 0.5136 μm
    "num_periods": [18.0, 26.0],    # Centered near N = 22 teeth
    "source_x": [3.500, 4.200]      # Centered near Fiber X = 3.8419 μm
}

In [ ]:
"""
DEV NOTE: I'd sincerely suggest using the budget function from the second
demo. I'd especially encourage it if you're looking for the neighbourhood of
the optimum, instead of the optimal value itself.

Any experimental defects will offset the device enough to make the optimum
unreachable anyway.
"""
# Output directory for chains and verification plots
demo_1_out = Path("example_results/bayesian_demo_unbounded")

# Initialize the Optimizer Engine
optimizer_unbounded = GratingCouplerBayesianOptimizer(
    fixed_params=fixed_params,       # Dict containing fixed structural/simulation parameters
    materials=materials,             # Dict containing Tidy3D material mediums
    output_dir=demo_1_out,           # Target path where HDF5 cache, chains, and plots are saved
    param_bounds=tight_param_bounds, # Tight prior bounds around known optimum
    sigma=0.02,                      # Target variance for Gaussian likelihood (0.02 = 2% tolerance)
    is_3d=False                      # Set to False for fast 2D FDTD, True for full 3D
)

# Execute Unbounded Nested Sampling
# Runs until evidence tolerance (ΔZ) falls below threshold without financial stopping limits
results_unbounded = optimizer_unbounded.execute_search(
    n_live_points=25,         # Number of active MultiNest live points (small for fast demo)
    evidence_tolerance=0.5    # Stopping tolerance for logarithmic evidence (ΔZ)
)

print(f"Max Coupling Efficiency Reached: {results_unbounded['max_efficiency']*100:.2f}%")
print(f"Total FlexCredits Spent: {results_unbounded['total_credits_used']:.4f}")

In [ ]:
# Output directory for chains, budget ledger, and leaderboard
demo_2_out = Path("example_results/bayesian_demo_budgeted")

# Initialize the Budget-Aware Optimizer Engine
optimizer_budgeted = GratingCouplerBayesianOptimizer(
    fixed_params=fixed_params,
    materials=materials,
    output_dir=demo_2_out,
    param_bounds=tight_param_bounds,
    sigma=0.02,
    is_3d=False,
    credit_limit=0.5   # Pre-set hard credit cap at initialization (optional)
)

# Execute Budget-Capped Nested Sampling
# Intercepts executions before cloud submission if (spent + next_run_cost) > credit_limit
results_budgeted = optimizer_budgeted.execute_search_with_budget(
    credit_limit=0.5,         # Hard cap: Max FlexCredits allowed for this entire run session
    n_live_points=25,          # Number of active MultiNest live points
    evidence_tolerance=0.5     # Convergence tolerance (if budget isn't exhausted first)
)

# Check run termination status
if results_budgeted.get("status") == "PAUSED_BUDGET_EXCEEDED":
    print(f"\n[Run Paused] Spent {results_budgeted['total_credits_used']:.4f} / {results_budgeted['credit_limit']:.4f} credits.")
    print("Pass a higher credit_limit to this directory to resume MultiNest seamlessly.")
else:
    print(f"\n[Run Converged] Max Efficiency: {results_budgeted['max_efficiency']*100:.2f}%")

# Print the Top 10 configurations collected before hitting the budget cap
optimizer_budgeted.report_top_configurations()

[Bayesian Engine] Custom search envelopes registered for 2D workspace.

[Bayesian Engine] Launching budget-aware MultiNest space extraction (2D)...
[Budget Warden] Active Cap: 0.5000 FlexCredits | Spent So Far: 0.0000 FlexCredits
 MultiNest Warning: no resume file found, starting from scratch
 *****************************************************
 MultiNest v3.10
 Copyright Farhan Feroz & Mike Hobson
 Release Jul 2015

 no. of live points =   25
 dimensionality =    5
 *****************************************************
 Starting MultiNest
 generating live points


03:10:29 IST Created task 'gc_eval_2D_p_0.8339_dc_0.4942_e_0' with resource_id  
             'fdve-60784b48-5be5-4788-88ec-e1ef57c77434' and task_type 'FDTD'.

             View task using web UI at                                          
             ]8;id=3570497;https://tidy3d.simulation.cloud/workbench?taskId=fdve-60784b48-5be5-4788-88ec-e1ef57c77434\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=3570498;https://tidy3d.simulation.cloud/workbench?taskId=fdve-60784b48-5be5-4788-88ec-e1ef57c77434\taskId]8;;\]8;id=3570497;https://tidy3d.simulation.cloud/workbench?taskId=fdve-60784b48-5be5-4788-88ec-e1ef57c77434\=]8;;\]8;id=3570499;https://tidy3d.simulation.cloud/workbench?taskId=fdve-60784b48-5be5-4788-88ec-e1ef57c77434\fdve]8;;\]8;id=3570497;https://tidy3d.simulation.cloud/workbench?taskId=fdve-60784b48-5be5-4788-88ec-e1ef57c77434\-60784b48-5be]8;;\
             ]8;id=3570497;https://tidy3d.simulation.cloud/workbench?taskId=fdve-60784b48-5be5-4788-88ec-e1ef57c77434\5-4788-88ec-e1ef57c77434']8;;\.

             Task folder: ]8;id=3570502;https://tidy3d.simulation.cloud/folders/folder-31ab8ed9-04f8-4aab-bdd9-855af6d8b115\'default']8;;\.

/home/esraaj/miniforge3/envs/tarang-env/lib/python3.11/site-packages/rich/live.p
y:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

03:10:34 IST Estimated FlexCredit cost: 0.025. Minimum cost depends on task     
             execution details. Use 'web.real_cost(task_id)' to get the billed  
             FlexCredit cost after a simulation run.

... (20 outputs left out)

03:11:14 IST Created task 'gc_eval_2D_p_0.8227_dc_0.5022_e_0' with resource_id  
             'fdve-62be404a-5899-4c3a-8f3b-43b163546490' and task_type 'FDTD'.

             View task using web UI at                                          
             ]8;id=3570527;https://tidy3d.simulation.cloud/workbench?taskId=fdve-62be404a-5899-4c3a-8f3b-43b163546490\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=3570528;https://tidy3d.simulation.cloud/workbench?taskId=fdve-62be404a-5899-4c3a-8f3b-43b163546490\taskId]8;;\]8;id=3570527;https://tidy3d.simulation.cloud/workbench?taskId=fdve-62be404a-5899-4c3a-8f3b-43b163546490\=]8;;\]8;id=3570529;https://tidy3d.simulation.cloud/workbench?taskId=fdve-62be404a-5899-4c3a-8f3b-43b163546490\fdve]8;;\]8;id=3570527;https://tidy3d.simulation.cloud/workbench?taskId=fdve-62be404a-5899-4c3a-8f3b-43b163546490\-62be404a-589]8;;\
             ]8;id=3570527;https://tidy3d.simulation.cloud/workbench?taskId=fdve-62be404a-5899-4c3a-8f3b-43b163546490\9-4c3a-8f3b-43b163546490']8;;\.

             Task folder: ]8;id=3570531;https://tidy3d.simulation.cloud/folders/folder-31ab8ed9-04f8-4aab-bdd9-855af6d8b115\'default']8;;\.

/home/esraaj/miniforge3/envs/tarang-env/lib/python3.11/site-packages/rich/live.p
y:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')